# GTM Agent — Weekly Signal Analysis

**Phase 1 Learning Loop** — Run this notebook weekly (after ≥ 3 sessions).

Reads `signal_snapshots` and `override_events` from SQLite to evaluate:
- Tier 1 precision and recall
- Which signals correlate with user keep/remove decisions
- Override rate by tier
- Remove reason distribution

**Trigger for Phase 2:** When labeled samples ≥ 80, run `scripts/train_scoring_model.py`.

In [ ]:
import sys
import os
import json
import sqlite3
import pandas as pd

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

DB_PATH = '../data/signal_store/gtm_agent.db'
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

## 1 — Load Data

In [ ]:
snapshots = pd.read_sql("""
    SELECT session_id, company_id, company_name, domain,
           tier_assigned, total_score,
           firmographic_score, keyword_score, growth_score,
           timing_score, lookalike_score,
           user_outcome, created_at
    FROM signal_snapshots
    ORDER BY created_at DESC
""", conn)

overrides = pd.read_sql("""
    SELECT session_id, company_id, action, remove_reason,
           score_breakdown_at_override, created_at
    FROM override_events
    ORDER BY created_at DESC
""", conn)

print(f"signal_snapshots rows: {len(snapshots)}")
print(f"override_events rows:  {len(overrides)}")
print(f"Labeled samples (user_outcome set): {snapshots['user_outcome'].notna().sum()}")

## 2 — Tier 1 Precision & Recall

**Precision:** % of Tier 1 companies that user kept  
**Recall:** % of kept companies that were Tier 1  

Targets: Precision > 70%, Recall > 60%. Alert if Precision < 50%.

In [ ]:
merged = pd.merge(
    snapshots,
    overrides[['session_id', 'company_id', 'action']],
    on=['session_id', 'company_id'],
    how='left'
)

labeled = merged[merged['action'].notna()]

if len(labeled) == 0:
    print("No labeled samples yet. Take actions (keep/remove) after a pipeline run to populate override_events.")
else:
    tier1 = labeled[labeled['tier_assigned'] == 'Tier 1']
    kept = labeled[labeled['action'] == 'keep']

    precision = (len(tier1[tier1['action'] == 'keep']) / len(tier1)) if len(tier1) > 0 else None
    recall = (len(kept[kept['tier_assigned'] == 'Tier 1']) / len(kept)) if len(kept) > 0 else None

    print(f"Tier 1 Precision : {precision:.1%}" if precision is not None else "Tier 1 Precision : n/a")
    print(f"Tier 1 Recall    : {recall:.1%}" if recall is not None else "Tier 1 Recall    : n/a")

    if precision is not None and precision < 0.50:
        print("⚠️  ALERT: Tier 1 Precision < 50% — scoring weights may need adjustment.")

## 3 — Override Rate by Tier

In [ ]:
if len(labeled) == 0:
    print("No labeled samples yet.")
else:
    override_rate = labeled.groupby('tier_assigned').apply(
        lambda g: (g['action'] == 'remove').sum() / len(g)
    ).rename('remove_rate')

    print("Override (remove) rate by tier:")
    print(override_rate.to_string())

    tier1_rate = override_rate.get('Tier 1', 0)
    if tier1_rate > 0.50:
        print(f"\n⚠️  ALERT: Tier 1 remove rate {tier1_rate:.0%} > 50% — consider raising tier_1_min threshold.")

## 4 — Remove Reason Distribution

If 'Wrong industry' > 30% → ICP parsing problem.  
If 'Too big' > 30% → company_size_max may be set too high.

In [ ]:
removed = overrides[overrides['action'] == 'remove']

if len(removed) == 0:
    print("No remove events yet.")
else:
    reason_dist = removed['remove_reason'].value_counts(normalize=True)
    print("Remove reason distribution:")
    print(reason_dist.to_string())

    top_reason_pct = reason_dist.iloc[0] if len(reason_dist) > 0 else 0
    top_reason = reason_dist.index[0] if len(reason_dist) > 0 else ''
    if top_reason_pct > 0.30:
        print(f"\n⚠️  '{top_reason}' accounts for {top_reason_pct:.0%} of removes.")

## 5 — Signal Contribution Analysis

Which scoring signals actually predict user keep/remove decisions?  
This is the direct input to Phase 2 weight retraining.

In [ ]:
if len(overrides) == 0:
    print("No override_events yet — no signal contribution data available.")
else:
    def extract_breakdown(row):
        try:
            return json.loads(row) if isinstance(row, str) else {}
        except (json.JSONDecodeError, TypeError):
            return {}

    breakdown_df = overrides['score_breakdown_at_override'].apply(extract_breakdown).apply(pd.Series)
    breakdown_df['kept'] = (overrides['action'] == 'keep').astype(int).values

    signal_cols = [c for c in breakdown_df.columns if c.endswith('_score')]
    if signal_cols:
        corr = breakdown_df[signal_cols + ['kept']].corr()['kept'].drop('kept').sort_values(ascending=False)
        print("Signal correlation with user keeping a company (higher = more predictive):")
        print(corr.to_string())
    else:
        print("Score breakdown data not yet available in override_events.")

## 6 — Phase 2 Readiness Check

In [ ]:
labeled_count = snapshots['user_outcome'].notna().sum()
override_count = len(overrides)
keep_count = len(overrides[overrides['action'] == 'keep'])
remove_count = len(overrides[overrides['action'] == 'remove'])

PHASE2_THRESHOLD = 80
PHASE2_MIN_POSITIVE = 20
PHASE2_MIN_NEGATIVE = 15

print("Phase 2 Training Readiness:")
print(f"  Total labeled samples : {override_count} / {PHASE2_THRESHOLD} needed")
print(f"  Keep (positive) labels: {keep_count} / {PHASE2_MIN_POSITIVE} needed")
print(f"  Remove (negative) labels: {remove_count} / {PHASE2_MIN_NEGATIVE} needed")

ready = override_count >= PHASE2_THRESHOLD and keep_count >= PHASE2_MIN_POSITIVE and remove_count >= PHASE2_MIN_NEGATIVE
if ready:
    print("\n✅ Ready for Phase 2! Run: python scripts/train_scoring_model.py")
else:
    remaining = PHASE2_THRESHOLD - override_count
    print(f"\n  {remaining} more labeled samples needed before Phase 2.")

In [ ]:
conn.close()